# Datathon@IndoML 2026 — Track 2: Noise Event Removal

Suppress noise events in Indic speech and submit enhanced audio plus transcripts.
Ends with `submission_track2.zip`.

## The submission has two artifacts

```
submission_track2.zip
├── transcripts.jsonl          <- transcript of YOUR enhanced audio, mandated ASR
├── vaani_eval_001.wav         <- 16 kHz mono PCM-16, same duration as the original
└── ...
```

Everything at the ZIP **root**. `transcripts.jsonl` is one JSON object per line:
`{"clip_id": "vaani_eval_001", "text": "..."}`, where `clip_id` is the filename without `.wav`.

**Submitting audio alone is scored as if every transcript were wrong.** A missing or empty
transcript counts as WER = 1.0 for that clip, and since `WER_noisy` is a fixed organiser
baseline of roughly 0.31:

```
dWER = 0.31 - 1.0 = -0.69  ->  -69%
```

That is exactly what an early submission of ours scored. It was never a modelling failure.

## Scoring, and what it implies

```
Combined = SI-SDR(synthetic subset only) + 100 x dWER(all clips)
```

- **SI-SDR** is computed only on the 4 h of synthetic clips, the only ones with a clean
  reference. It is reported as an **absolute** value, not an improvement.
- **dWER** is computed on all 11 h, from your submitted transcripts against private ground truth.
- An **audit** re-runs the mandated ASR on a hidden subset of your WAVs and compares. Using a
  different ASR, or editing transcripts, fails it.

The leaderboard leader sits at SI-SDR 16.14 with dWER +0.44. **Combined is dominated by
SI-SDR.** Treat dWER as a penalty to keep at or above zero, and spend the effort on SI-SDR.

## Approach

| stage | choice | why |
|---|---|---|
| model | BiGRU magnitude-mask net on the STFT, reconstructed with the mixture phase | small enough to converge on the data available |
| encoder | **SraVaani-1.0** FastConformer features via FiLM | in-domain (31k h of Vaani) and ASR-aligned, which is the right bias when half the metric is word error |
| loss | **SI-SDR in the time domain** | the leaderboard metric itself, not a proxy |
| training | simulated mixtures, then **fine-tuned on the real validation pairs** | our pseudo-clean bank is not clean; the organisers' pairs are |
| output | blend with the mixture, then **restore RMS** | SI-SDR is scale-invariant, ASR is not |

## Setup

1. Accept terms on the Vaani dataset **and** on `ARTPARK-IISc/SraVaani-1.0` (both gated).
2. Kaggle -> Add-ons -> Secrets -> `HF_TOKEN`. Internet **ON**, GPU on.
3. Attach `indoml-validation` and the Track 2 test audio as Kaggle Datasets.

---
## Architecture at a glance

Think of the model as a **smart volume knob**. It looks at noisy speech and decides, for every
tiny slice of time and pitch, **how much to turn down**. It never invents new speech — it only
keeps or removes what is already there.

### The pipeline in one picture

Read the diagram top-to-bottom. The note to the **right of every box** says what that box
does and *why* it is there.

```
  noisy speech                        ◄─ the input: one Vaani clip, speech with noise events mixed in
       │
       ▼
  ┌──────────────────────────┐
  │  1. MAKE PAIRED DATA      │       ◄─ PROBLEM: to grade a denoiser you need the clean answer,
  │        (§3–§4)            │          but the dataset never gives clean/noisy pairs. So we
  │                          │          BUILD them from the raw clips using the event time-stamps:
  │   clean = speech between  │       ◄─ cut the gaps between noise events → "clean" speech
  │           the events      │
  │   noise = the events      │       ◄─ cut the events themselves (a VAD drops chunks that are
  │           themselves      │          actually speech, so the noise is really noise)
  │   mix   = clean + noise   │       ◄─ add them back together OURSELVES, so we know the exact
  │                          │          clean signal that produced this mix = the perfect target
  └────────────┬─────────────┘
       │
       ▼
  ┌──────────────────────────┐
  │     2. THE DENOISER       │       ◄─ the neural network being trained (§5)
  │           (§5)            │
  │                          │
  │  noisy ─► STFT picture    │       ◄─ turn the 1-D waveform into a 2-D time × pitch spectrogram;
  │                          │          noise and speech light up different squares → separable
  │  SraVaani ASR ─┐ (frozen) │       ◄─ a pre-trained speech recogniser looks at the same clip and
  │                ▼          │          whispers "this part is a real word, don't erase it".
  │  BiGRU "brain" │          │          Frozen = never trained, only advises (protects the words)
  │                ▼          │       ◄─ the BiGRU reads the picture forwards AND backwards, using
  │  mask (0..1)   │          │          context to decide, for every square, how much to KEEP
  │                ▼          │          (1 = pure speech, 0 = pure noise, 0.3 = mostly noise)
  │  picture × mask ─► audio  │       ◄─ multiply the picture by the mask, then invert (iSTFT)
  └────────────┬─────────────┘          back to a clean waveform
       │
       │  how the denoiser LEARNS:
       │   • 3. train to maximise SI-SDR (§6) ◄─ the exact leaderboard metric is the loss
       │   • 4. fine-tune on real pairs  (§10)◄─ redo on the organisers' genuinely-clean
       │                                          pairs; closes the sim→real gap (biggest jump)
       ▼
  ┌──────────────────────────┐
  │   5. POLISH THE OUTPUT    │       ◄─ the mask can only turn sound DOWN, so the output comes
  │          (§7)            │          out too quiet. SI-SDR ignores loudness, but the ASR
  │  turn the volume back up  │          needs it → rescale the output to the input's loudness
  │  (RMS restore)            │          (free: costs 0 dB of SI-SDR, proven in §7)
  └────────────┬─────────────┘
       │
       ▼
  ┌──────────────────────────┐
  │  6. TRANSCRIBE (§8)       │       ◄─ run the MANDATED ASR on your enhanced audio to get the
  │                          │          transcript. An audit re-runs this exact model on your
  │                          │          WAVs, so you must use it unchanged
  └────────────┬─────────────┘
       │
       ▼
  ┌──────────────────────────┐
  │  7. PACK THE ZIP (§11)    │       ◄─ one WAV per clip + one transcripts.jsonl, all at the ZIP
  │                          │          root. A missing WAV or empty transcript is scored as
  │  submission_track2.zip    │          maximum error, so every file is asserted before zipping
  └──────────────────────────┘
```

### The five moving parts

| part | plain-English job |
|---|---|
| **make data (§3–§4)** | no clean/noisy pairs are given, so we cut clean speech and noise out of the raw clips and glue them back together — now we know the perfect answer |
| **the expert — SraVaani (§5)** | a frozen speech-recogniser whispers *"that's a real word, don't erase it"* so the model doesn't damage speech |
| **the brain — BiGRU (§5)** | reads the sound forwards and backwards and outputs the mask: how much to keep at every moment/pitch |
| **the lesson — SI-SDR loss (§6)** | trains the model on the exact score the leaderboard uses |
| **the polish (§7)** | the mask only turns things *down*, so the output is too quiet — we turn the volume back up (free points, and the ASR needs it) |

### The one thing to remember

**SI-SDR is most of the score; word-error is a penalty to keep from going negative.** So the
effort goes into the denoiser and the fine-tune, while the polish and transcription steps mostly
exist so a silly mistake (too quiet, missing transcript) doesn't tank the score.

---
## 0. Config

In [ ]:
import os, json, math, random, time, io, sys, glob, zipfile, shutil, re, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np

BUDGET = "tiny"                 # "tiny" smoke test (~100 clips), "fast" ~1h30, "full" ~3h

CFG = dict(sr=16000, clip_sec=5.0, seed=42)
N_FFT, HOP = 512, 128

# ---- exact submission spec (from the Evaluation page) ----
TRANSCRIPT_NAME = "transcripts.jsonl"
ID_KEY, TEXT_KEY = "clip_id", "text"          # clip_id = filename WITHOUT .wav
WAV_SUBTYPE = "PCM_16"                        # WAV, PCM 16-bit signed
ASR_REPO = "ARTPARK-IISc/SraVaani-1.0"        # mandated; the audit checks this

# ---- output stage ----
BLEND = 0.85                                  # y = BLEND*est + (1-BLEND)*mix
MATCH_RMS = True                              # restore level; free under SI-SDR, vital for ASR

T2_ENCODER = "sravaani"                       # "sravaani" | "wavlm" | "none"
WAVLM_NAME = "microsoft/wavlm-base-plus"
ENC_PROJ = 128

if BUDGET == "tiny":
    MAX_GOLD, N_MIX, EPOCHS, BS = 100, 100, 3, 8
elif BUDGET == "fast":
    MAX_GOLD, N_MIX, EPOCHS, BS = 6000, 4000, 25, 8
else:
    MAX_GOLD, N_MIX, EPOCHS, BS = 12000, 8000, 40, 8

REPO = "ARTPARK-IISc/Vaani-Noise-Event-Dataset"
WORK = Path("/kaggle/working")

CLEAN_DIR = WORK/"bank/clean"; NOISE_DIR = WORK/"bank/noise"
MIX_DIR = WORK/"t2/mix"; REF_DIR = WORK/"t2/clean"
MANIFEST = WORK/"t2_manifest.jsonl"; CKPT = WORK/"t2_model.pt"
for d in [CLEAN_DIR, NOISE_DIR, MIX_DIR, REF_DIR]:
    try: d.mkdir(parents=True, exist_ok=True)
    except Exception: pass

SIM = dict(dur_sec=CFG["clip_sec"], min_clean_sec=2.5, min_noise_sec=0.3, max_noise_sec=3.0,
           events=(1, 2), snr_db=(-5.0, 12.0), seed=1234)
MAX_SPEECH_RATIO = 0.35
N_SAMP = int(CFG["clip_sec"]*CFG["sr"])
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print(f"BUDGET={BUDGET} | encoder {T2_ENCODER} | blend {BLEND} | match_rms {MATCH_RMS}")

---
## 1. The official scorer, verbatim

Copied from the Evaluation page so the local number equals the leaderboard number.

Two details that are easy to get wrong. dWER is **pooled** — total edits over total reference
words via `jiwer.wer` on lists — not a mean of per-clip ratios; averaging ratios lets one ASR
repetition loop dominate a whole subset. And `normalize()` strips `<...>` and `[...]`, which
matters because Vaani transcripts carry inline noise tags like `<horn> ... </horn>`.

In [ ]:
!pip -q install jiwer sentencepiece 2>/dev/null | tail -1
from jiwer import wer as jiwer_wer

def si_sdr(reference, enhanced):
    ref = np.asarray(reference, dtype=np.float64)
    enh = np.asarray(enhanced, dtype=np.float64)
    n = min(len(ref), len(enh)); ref, enh = ref[:n], enh[:n]
    scale = np.dot(enh, ref) / np.dot(ref, ref)
    s_target = scale * ref
    e_noise = enh - s_target
    value = 10.0*np.log10(np.dot(s_target, s_target) / np.dot(e_noise, e_noise))
    return float(np.clip(value, -100.0, 100.0))

TAG_RE = re.compile(r"</?[^<>]*>|\[[^\[\]]*\]")

def normalize(text):
    s = TAG_RE.sub(" ", text or "")
    s = "".join(" " if unicodedata.category(c).startswith("P") else c for c in s)
    return " ".join(s.lower().split())

def delta_wer(gt, noisy_asr, submitted, clip_ids):
    refs, noisy, enh = [], [], []
    for cid in clip_ids:
        g = normalize(gt[cid])
        if not g: continue
        refs.append(g)
        noisy.append(normalize(noisy_asr.get(cid, "")) or "@")
        enh.append(normalize(submitted.get(cid, "")) or "@")
    return jiwer_wer(refs, noisy) - jiwer_wer(refs, enh)      # fraction; x100 = percent

def combined(si_sdr_synth, dwer_fraction):
    return si_sdr_synth + 100.0*dwer_fraction

# the -69 reproduced from the formula alone: every transcript missing -> WER 1.0 everywhere
print(f"missing transcripts, WER_noisy=0.31 -> dWER {100*(0.31-1.0):+.1f} "
      f"(you scored -69.35)")

---
## 2. Validation set

`indoml-validation` ships as three folders, and they map directly onto the two metrics:

| folder | role |
|---|---|
| `naturalNoisyAudio/` | dWER only — no clean reference exists for natural clips |
| `syntheticNoiseAudio/` | dWER **and** SI-SDR |
| `syntheticCleanRefAudio/` | the clean reference, matched by filename stem |

`validationMetadata.json` is `{"root": [ ... ]}` with `segmentFileName` as the id and
`transcript` as ground truth. Real ground truth means **dWER here is a real measurement**, not
a proxy.

This is also the only genuinely paired data available, which makes it both the evaluation set
and — in Section 10 — the fine-tuning set. Section 10 holds out a slice to keep that honest.

In [ ]:
AUD = (".wav", ".flac", ".mp3", ".ogg")

def scan_inputs():
    root = Path("/kaggle/input")
    if not root.exists():
        print("no /kaggle/input"); return
    for d in sorted(root.iterdir()):
        n_aud = sum(1 for p in d.rglob("*") if p.suffix.lower() in AUD)
        n_pq = sum(1 for p in d.rglob("*.parquet"))
        others = [p for p in d.rglob("*") if p.is_file() and p.suffix.lower() not in AUD]
        print(f"\n=== {d.name} : {n_aud} audio, {n_pq} parquet ===")
        for p in sorted(others)[:12]:
            print(f"   {p.relative_to(d)}  ({p.stat().st_size} B)")

def biggest_audio_dir(exclude=()):
    root = Path("/kaggle/input"); best = None
    for d in (sorted(root.iterdir()) if root.exists() else []):
        if d in exclude:
            continue
        n = sum(1 for p in d.rglob("*") if p.suffix.lower() in AUD)
        if n and (best is None or n > best[1]):
            best = (d, n)
    return best[0] if best else None

scan_inputs()
print("\nValidation is loaded in the next cell; TEST_DIR is resolved in Section 10.")

In [ ]:
# ============ VALIDATION: indoml-validation folder layout ============
#   validation/
#     naturalNoisyAudio/        natural clips  -> dWER only (no clean reference exists)
#     syntheticNoiseAudio/      synthetic noisy -> dWER AND SI-SDR
#     syntheticCleanRefAudio/   clean reference for the synthetic clips
#     validationMetadata.json
AUD = (".wav", ".flac", ".mp3", ".ogg")

def _find_val_root():
    for c in [Path("/kaggle/input/indoml-validation"),
              Path("/kaggle/input/datasets/srinjoy3222/indoml-validation")]:
        if c.exists(): return c
    root = Path("/kaggle/input")
    for d in (sorted(root.iterdir()) if root.exists() else []):
        if any("validation" in q.name.lower() for q in d.rglob("*") if q.is_dir()):
            return d
    return None

VAL_DIR = _find_val_root()
assert VAL_DIR is not None, "indoml-validation not found under /kaggle/input"
print("VAL_DIR =", VAL_DIR)

def _dir_like(*keys):
    for d in VAL_DIR.rglob("*"):
        if d.is_dir() and all(k.lower() in d.name.lower() for k in keys):
            return d
    return None

NAT_DIR   = _dir_like("natural", "noisy")
SYN_DIR   = _dir_like("synthetic", "noise")
CLEAN_REF = _dir_like("synthetic", "clean")
for nm, d in [("naturalNoisyAudio", NAT_DIR), ("syntheticNoiseAudio", SYN_DIR),
              ("syntheticCleanRefAudio", CLEAN_REF)]:
    n = len([p for p in d.rglob("*") if p.suffix.lower() in AUD]) if d else 0
    print(f"  {nm:<24} {'-' if d is None else d.name:<28} {n} files")

# ---- metadata ----
val_meta = {}
mp = next((p for p in VAL_DIR.rglob("*.json") if "metadata" in p.name.lower()), None)
if mp is None:
    mp = next(iter(VAL_DIR.rglob("*.json")), None)
if mp is not None:
    print(f"\nmetadata: {mp.name}")
    obj = json.loads(mp.read_text(encoding="utf-8", errors="replace"))
    def _strip(k):
        k = str(k)
        return k[:-4] if k.lower().endswith(".wav") else k
    if isinstance(obj, dict) and obj and all(isinstance(v, dict) for v in obj.values()):
        val_meta = {_strip(k): v for k, v in obj.items()}          # keyed by clip id
    else:
        rows = obj if isinstance(obj, list) else [obj]
        for key in ("clips", "data", "items", "records", "validation"):
            if isinstance(obj, dict) and isinstance(obj.get(key), list):
                rows = obj[key]; break
        for r in rows:
            if not isinstance(r, dict): continue
            cid = (r.get("clip_id") or r.get("id") or r.get("filename")
                   or r.get("audio") or r.get("file"))
            if cid is not None:
                val_meta[_strip(cid)] = r
    print(f"  {len(val_meta)} entries")
    if val_meta:
        k0 = next(iter(val_meta))
        print(f"  keys: {sorted(val_meta[k0])}")
        print(f"  sample[{k0}]: {json.dumps(val_meta[k0], ensure_ascii=False)[:260]}")

def _pick(r, *names):
    for n in names:
        v = r.get(n)
        if v not in (None, "", [], {}): return v
    return None

# ---- build the item list ----
def _ref_for(stem):
    if CLEAN_REF is None: return None
    for q in CLEAN_REF.rglob(f"{stem}.*"):
        if q.suffix.lower() in AUD: return q
    base = re.sub(r"(_noisy|_noise|_mix|_mixed)$", "", stem, flags=re.I)
    for q in CLEAN_REF.rglob(f"{base}.*"):
        if q.suffix.lower() in AUD: return q
    return None

val_items = []
for d, syn in [(NAT_DIR, False), (SYN_DIR, True)]:
    if d is None: continue
    for p in sorted(q for q in d.rglob("*") if q.suffix.lower() in AUD):
        r = val_meta.get(p.stem, {})
        val_items.append(dict(id=p.stem, path=p, syn=syn, ref=(_ref_for(p.stem) if syn else None),
                              gt=_pick(r, "transcript", "text", "ground_truth",
                                       "reference_text", "sentence")))

n_syn = sum(1 for v in val_items if v["syn"])
n_ref = sum(1 for v in val_items if v["ref"] is not None)
n_gt  = sum(1 for v in val_items if v["gt"])
print(f"\n{len(val_items)} clips | synthetic {n_syn} | natural {len(val_items)-n_syn} "
      f"| clean ref matched {n_ref} | GT transcript {n_gt}")
print(f"SI-SDR locally computable : {'YES on ' + str(n_ref) + ' synthetic clips' if n_ref else 'NO'}")
print(f"dWER   locally computable : {'YES (true GT)' if n_gt else 'proxy via ASR on clean ref'}")
if n_syn and not n_ref:
    print("!! synthetic clips found but no reference matched - stems differ between folders.")
    print("   syn sample :", [p.stem for p in list(SYN_DIR.rglob('*'))[:3] if p.suffix.lower() in AUD])
    print("   ref sample :", [p.stem for p in list(CLEAN_REF.rglob('*'))[:3] if p.suffix.lower() in AUD])

---
## 3. Training data — clean and noise banks

SI-SDR training needs paired data, and the released training set has none, so we build it.

**Clean** = the complement of annotated event spans in Gold clips: Vaani speech with no
annotated noise. That is *pseudo*-clean, not clean — real background survives, and this is the
domain gap Section 10 addresses. Say so in your report.

**Noise** = the event spans themselves, filtered by a VAD. Without that filter roughly half of
each "noise" span is speech, which would make the SI-SDR target wrong and teach the model to
delete words.

In [ ]:
import torch, soundfile as sf, librosa
import torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(CFG["seed"]); torch.cuda.manual_seed_all(CFG["seed"])

def read_audio(p):
    w, sr = sf.read(str(p), dtype="float32", always_2d=False)
    if w.ndim > 1: w = w.mean(axis=1)
    if sr != CFG["sr"]: w = librosa.resample(w, orig_sr=sr, target_sr=CFG["sr"])
    return np.ascontiguousarray(w, dtype=np.float32)

def decode_audio(a, target_sr=CFG["sr"]):
    w = sr = None
    if isinstance(a, dict):
        if a.get("array") is not None:
            w, sr = np.asarray(a["array"], dtype=np.float32), a["sampling_rate"]
        elif a.get("bytes"):
            w, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32", always_2d=False)
        elif a.get("path"):
            w, sr = sf.read(a["path"], dtype="float32", always_2d=False)
    elif hasattr(a, "get_all_samples"):
        s = a.get_all_samples(); w, sr = s.data.numpy().astype(np.float32), int(s.sample_rate)
    if w is None: raise ValueError(f"cannot decode {type(a)}")
    w = np.asarray(w, dtype=np.float32)
    if w.ndim > 1:
        w = w.mean(axis=0) if w.shape[0] < w.shape[1] else w.mean(axis=1)
    if sr != target_sr: w = librosa.resample(w, orig_sr=sr, target_sr=target_sr)
    return np.ascontiguousarray(w, dtype=np.float32)

def _f(x):
    try: return float(x)
    except: return None

def spans_from(ex):
    out = []
    for s in (ex.get("NoiseSubCategoryTimeStamp") or []):
        st, en = _f(s.get("start")), _f(s.get("end"))
        if st is not None and en is not None and en > st: out.append((st, en))
    return sorted(out)

def complement(spans, dur, pad=0.05):
    free, cur = [], 0.0
    for st, en in spans:
        st, en = max(0.0, st-pad), min(dur, en+pad)
        if st > cur: free.append((cur, st))
        cur = max(cur, en)
    if cur < dur: free.append((cur, dur))
    return free

try:
    import webrtcvad; _vad = webrtcvad.Vad(2); HAVE_VAD = True
except Exception:
    HAVE_VAD = False

def speech_ratio(w, sr=CFG["sr"]):
    if HAVE_VAD:
        pcm = np.clip(w*32767, -32768, 32767).astype(np.int16).tobytes()
        n = int(sr*0.03)*2
        fr = [pcm[i:i+n] for i in range(0, len(pcm)-n+1, n)]
        return float(np.mean([_vad.is_speech(f, sr) for f in fr])) if fr else 1.0
    S = np.abs(np.fft.rfft(w*np.hanning(len(w))))**2
    f = np.fft.rfftfreq(len(w), 1/sr)
    return float(S[(f >= 300) & (f <= 3400)].sum()/(S.sum()+1e-9))

In [ ]:
!pip -q install datasets huggingface_hub soundfile librosa webrtcvad-wheels transformers 2>/dev/null | tail -1
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Audio
from tqdm.auto import tqdm

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

# Stream the dataset: iterate rows directly, no shard scanning or manual parquet picking.
# The next cell reads through `raw`, keeps the Gold clips, and stops once MAX_GOLD is hit,
# so streaming means we never download shards we won't use.
raw = load_dataset(REPO, split="train", streaming=True, token=HF_TOKEN)
# decode=False hands back raw bytes we decode ourselves (version-proof across `datasets`).
raw = raw.cast_column("audio", Audio(decode=False))
print("dataset ready (streaming, decode=False)")


In [ ]:
clean_idx, noise_idx, n_gold = [], [], 0
t0 = time.time()
for i, ex in enumerate(tqdm(raw, desc="banks")):
    if ex["annotationQuality"] != "verified_timestamps": continue
    if n_gold >= MAX_GOLD: break
    try: wav = decode_audio(ex["audio"])
    except Exception: continue
    if wav is None or len(wav) < CFG["sr"]*0.5: continue
    n_gold += 1
    dur = len(wav)/CFG["sr"]; sp = spans_from(ex)
    for k, (a, b) in enumerate(complement(sp, dur)):
        if b-a < SIM["min_clean_sec"]: continue
        seg = wav[int(a*CFG["sr"]):int(b*CFG["sr"])]
        if np.abs(seg).max() < 1e-3: continue
        p = CLEAN_DIR/f"c{i:07d}_{k}.wav"; sf.write(p, seg, CFG["sr"])
        clean_idx.append({"path": str(p), "dur": len(seg)/CFG["sr"]})
    for k, (a, b) in enumerate(sp):
        d = min(b, dur)-max(0.0, a)
        if not (SIM["min_noise_sec"] <= d <= SIM["max_noise_sec"]): continue
        seg = wav[int(a*CFG["sr"]):int(b*CFG["sr"])]
        if np.abs(seg).max() < 1e-3 or speech_ratio(seg) > MAX_SPEECH_RATIO: continue
        p = NOISE_DIR/f"n{i:07d}_{k}.wav"; sf.write(p, seg, CFG["sr"])
        noise_idx.append({"path": str(p), "dur": len(seg)/CFG["sr"]})
json.dump(clean_idx, open(WORK/"clean_index.json", "w"))
json.dump(noise_idx, open(WORK/"noise_index.json", "w"))
print(f"{n_gold} gold clips in {(time.time()-t0)/60:.1f} min")

print(f"clean {len(clean_idx)} ({sum(c['dur'] for c in clean_idx)/3600:.2f} h) | "
      f"noise {len(noise_idx)} ({sum(n['dur'] for n in noise_idx)/3600:.2f} h)")
assert clean_idx and noise_idx, "empty bank"
d = np.array([c["dur"] for c in clean_idx])
print("clean span p10/p50/p90: %.2f %.2f %.2f s" % tuple(np.percentile(d, [10, 50, 90])))

---
## 4. Simulate mixtures

One clean span from **one** source clip — splicing spans from different clips would change
speaker and channel every couple of seconds, which is not speech any ASR should be scored on.

Events are scaled to a sampled **event-local SNR**, measured over the span where the event
sits. Global SNR is misleading when an event covers a fraction of the clip. The local power is
floored against the clip's overall power so an event landing on a pause is not scaled to silence.

The assertion at the end verifies `mix == clean + noise` outside the event spans. If that fails,
the simulation is not self-consistent and nothing downstream is trustworthy.

In [ ]:
rng = np.random.default_rng(SIM["seed"])

def _fit(w, n):
    if len(w) >= n:
        s = rng.integers(0, len(w)-n+1); return w[s:s+n]
    return np.tile(w, int(np.ceil(n/max(1, len(w)))))[:n]

def build_clean(idx, tries=30):
    best = None
    for _ in range(tries):
        c = idx[rng.integers(0, len(idx))]
        w, _ = sf.read(c["path"], dtype="float32")
        if w.ndim > 1: w = w.mean(axis=1)
        if best is None or len(w) > len(best): best = w
        if len(w) >= SIM["min_clean_sec"]*CFG["sr"]: return w[:N_SAMP]
    return best[:N_SAMP] if best is not None else None

def simulate_one():
    clean = build_clean(clean_idx)
    if clean is None or len(clean) < int(SIM["min_clean_sec"]*CFG["sr"]): return None, None, []
    n = len(clean); pk = np.abs(clean).max()
    if pk > 0: clean = clean/pk*rng.uniform(0.3, 0.85)
    mix = clean.copy(); ref_pow = float(np.mean(clean**2))
    placed, occ = [], []
    for _ in range(int(rng.integers(SIM["events"][0], SIM["events"][1]+1))):
        nb = noise_idx[rng.integers(0, len(noise_idx))]
        w, _ = sf.read(nb["path"], dtype="float32")
        if w.ndim > 1: w = w.mean(axis=1)
        L = int(min(len(w)/CFG["sr"], SIM["max_noise_sec"])*CFG["sr"])
        if L < int(SIM["min_noise_sec"]*CFG["sr"]) or L >= n: continue
        s = None
        for _t in range(20):
            cand = int(rng.integers(0, n-L))
            if all(cand+L <= a or cand >= b for a, b in occ): s = cand; break
        if s is None: continue
        snr = float(rng.uniform(*SIM["snr_db"])); seg = _fit(w, L)
        pc = max(float(np.mean(clean[s:s+L]**2)), 0.1*ref_pow, 1e-10)
        pn = float(np.mean(seg**2))+1e-10
        seg = seg*math.sqrt(pc/(pn*10**(snr/10.0)))
        f = min(int(0.02*CFG["sr"]), L//2)
        if f > 0:
            seg[:f] *= np.linspace(0, 1, f); seg[-f:] *= np.linspace(1, 0, f)
        mix[s:s+L] += seg; occ.append((s, s+L))
        placed.append({"onset": round(s/CFG["sr"], 3), "offset": round((s+L)/CFG["sr"], 3),
                       "snr_db": round(snr, 2)})
    m = np.abs(mix).max()
    if m > 0.99: mix, clean = mix/m*0.99, clean/m*0.99
    return mix.astype(np.float32), clean.astype(np.float32), placed

records = []
for i in tqdm(range(N_MIX), desc="simulate"):
    cid = f"sim{i:06d}"
    mix, clean, ev = simulate_one()
    if mix is None or not ev: continue
    sf.write(MIX_DIR/f"{cid}.wav", mix, CFG["sr"]); sf.write(REF_DIR/f"{cid}.wav", clean, CFG["sr"])
    records.append({"id": cid, "mix": str(MIX_DIR/f"{cid}.wav"),
                    "clean": str(REF_DIR/f"{cid}.wav"), "events": ev})
assert records, "no mixtures produced"
with open(MANIFEST, "w") as f:
    for r in records: f.write(json.dumps(r)+"\n")
print(len(records), "mixtures")

resid, sis = [], []
for r in records[:200]:
    mix, _ = sf.read(r["mix"], dtype="float32"); cl, _ = sf.read(r["clean"], dtype="float32")
    sis.append(si_sdr(cl, mix))
    inside = np.zeros(len(mix), bool)
    for e in r["events"]:
        a = max(0, int(e["onset"]*CFG["sr"])-16); b = min(len(mix), int(e["offset"]*CFG["sr"])+16)
        inside[a:b] = True
    if (~inside).any(): resid.append(np.abs((mix-cl)[~inside]).max())
print(f"mix==clean outside events: {max(resid):.2e} | input SI-SDR {np.mean(sis):+.2f} dB")
assert max(resid) < 1e-2

---
## 5. Dataset, encoder, model

**Encoder.** SraVaani-1.0 is loaded by reading its exported TorchScript graph directly —
`model-asr.fp16.ts` — bypassing the HF wrapper, the custom modeling code and SentencePiece. Its
`config.json` specifies `feat_in=128`, not the NeMo default of 80, and `preproc.pt` carries the
exact window and mel filterbank, so the front-end is reproduced rather than guessed. FastConformer
subsamples 8x, giving 80 ms frames, which are interpolated up to the STFT rate.

Why an ASR encoder here: dWER is half the metric and the dominant failure mode is
over-suppression deleting phonemes. Features that encode what is phonetically load-bearing are
the right restraint. (The same bias makes it a poor Track 1 encoder — an ASR model is trained to
be *invariant* to background events.)

**Model.** A BiGRU predicts a magnitude mask on the mixture spectrogram; the signal is
reconstructed with the mixture phase. Encoder features enter by **FiLM at two depths**, not
concatenated at the input — input-only conditioning has been measured to do worse than none at
all, because the signal is diluted before reaching the layers that use it.

`wavlm` and `none` are available as fallbacks so an encoder failure cannot block the run.

In [ ]:
from torch.utils.data import Dataset, DataLoader

_win = torch.hann_window(N_FFT).to(device)
def stft(x): return torch.stft(x, N_FFT, HOP, window=_win, return_complex=True)
def istft(X, n): return torch.istft(X, N_FFT, HOP, window=_win, length=n)

class T2Data(Dataset):
    def __init__(self, recs): self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        mix, _ = sf.read(r["mix"], dtype="float32"); clean, _ = sf.read(r["clean"], dtype="float32")
        mix, clean = mix[:N_SAMP], clean[:N_SAMP]
        if len(mix) < N_SAMP:
            mix = np.pad(mix, (0, N_SAMP-len(mix))); clean = np.pad(clean, (0, N_SAMP-len(clean)))
        return dict(mix=torch.from_numpy(mix.astype(np.float32)),
                    clean=torch.from_numpy(clean.astype(np.float32)))

split = int(0.95*len(records))
train_recs, sim_val_recs = records[:split], records[split:]
train_dl = DataLoader(T2Data(train_recs), batch_size=BS, shuffle=True, num_workers=2,
                      drop_last=True)
val_dl = DataLoader(T2Data(sim_val_recs), batch_size=BS, shuffle=False, num_workers=2)
b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()}, "| train", len(train_recs))

In [ ]:
class SraVaaniEnc(nn.Module):
    # loads the exported TorchScript graph directly: no HF wrapper, no tokenizer.
    # config.json says feat_in=128 (not the NeMo default 80); preproc.pt has the exact
    # window and filterbank, so the front end is reproduced rather than guessed.
    def __init__(self, repo=ASR_REPO, token=None):
        super().__init__()
        from huggingface_hub import snapshot_download
        path = snapshot_download(repo, token=token)
        ts = sorted(glob.glob(os.path.join(path, "*.ts")))
        if not ts: raise RuntimeError("no TorchScript graph")
        self.enc = torch.jit.load(ts[0], map_location="cpu").eval()
        self.pdtype = torch.float32
        for p in self.enc.parameters(): self.pdtype = p.dtype; break
        for p in self.enc.parameters(): p.requires_grad_(False)
        pp = torch.load(os.path.join(path, "preproc.pt"), map_location="cpu",
                        weights_only=False)
        self.register_buffer("win", pp["window"].float(), persistent=False)
        self.register_buffer("fb", pp["fb"].float(), persistent=False)
        prm = dict(pp.get("params", {}) or {})
        self.n_fft = int(prm.get("n_fft", 512))
        self.hop = int(prm.get("hop_length", prm.get("hop", 160)))
        self.preemph = float(prm.get("preemph", 0.97) or 0.0)
        self.guard = float(prm.get("log_zero_guard_value", 2**-24))
        self.norm_c = float(prm.get("normalize_constant", 1e-5))
        print(f"   {os.path.basename(ts[0])}: {self.fb.shape[0]} mels, n_fft {self.n_fft}, "
              f"hop {self.hop}, {self.pdtype}")
        self._call = None; self.eval()
    @staticmethod
    def _norm(x, seq_len, c):
        B, _, T = x.shape
        steps = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        valid = steps < seq_len.unsqueeze(1); denom = valid.sum(dim=1)
        mean = torch.where(valid.unsqueeze(1), x, torch.zeros_like(x)).sum(2)/denom.unsqueeze(1)
        var = torch.sum(torch.where(valid.unsqueeze(1), x-mean.unsqueeze(2),
                                    torch.zeros_like(x))**2, dim=2)/(denom.unsqueeze(1)-1.0)
        std = torch.sqrt(var); std = std.masked_fill(std.isnan(), 0.0)+c
        return (x-mean.unsqueeze(2))/std.unsqueeze(2)
    @torch.no_grad()
    def forward(self, wav):
        x = wav
        if self.preemph:
            x = torch.cat([x[:, :1], x[:, 1:]-self.preemph*x[:, :-1]], dim=1)
        X = torch.stft(x, self.n_fft, self.hop, win_length=self.win.numel(),
                       window=self.win.to(x.device), center=True, return_complex=True)
        mel = torch.log(torch.matmul(self.fb.to(x.device), X.abs()**2)+self.guard)
        ln = torch.full((mel.shape[0],), mel.shape[-1], device=mel.device, dtype=torch.float32)
        feats = self._norm(mel, ln, self.norm_c).to(self.pdtype)
        li = torch.full((feats.shape[0],), feats.shape[-1], dtype=torch.long, device=x.device)
        trials = [("kw", lambda: self.enc(audio_signal=feats, length=li)),
                  ("pos", lambda: self.enc(feats, li))]
        if self._call: trials = [t for t in trials if t[0] == self._call]
        err = []
        for name, fn in trials:
            try: out = fn()
            except Exception as e: err.append(f"{name}: {e}"); continue
            self._call = name
            out = out[0] if isinstance(out, (tuple, list)) else out
            return out.transpose(1, 2).float()
        raise RuntimeError("encoder call failed: "+" | ".join(err))

class WavLMEnc(nn.Module):
    def __init__(self, name=WAVLM_NAME):
        super().__init__()
        from transformers import AutoModel
        self.m = AutoModel.from_pretrained(name)
        for p in self.m.parameters(): p.requires_grad_(False)
        self.eval()
    @torch.no_grad()
    def forward(self, wav):
        x = (wav-wav.mean(-1, keepdim=True))/(wav.std(-1, keepdim=True)+1e-5)
        return self.m(x).last_hidden_state.float()

def build_encoder():
    if T2_ENCODER == "none":
        print("spectrogram only"); return None, 0
    order = ([("sravaani", SraVaaniEnc), ("wavlm", WavLMEnc)] if T2_ENCODER == "sravaani"
             else [("wavlm", WavLMEnc)])
    tok = HF_TOKEN if "HF_TOKEN" in globals() else None
    for name, cls in order:
        try:
            print(f"loading {name}...")
            e = (cls(token=tok) if name == "sravaani" else cls()).to(device)
            with torch.no_grad(): pr = e(torch.zeros(1, CFG["sr"], device=device))
            print(f"   OK 1.00 s -> {pr.shape[1]} frames "
                  f"({1000/pr.shape[1]:.0f} ms/frame) dim {pr.shape[-1]}")
            return e, pr.shape[-1]
        except Exception as ex:
            print(f"   {name} failed ({type(ex).__name__}: {str(ex)[:100]})")
    print("all encoders failed - spectrogram only")
    return None, 0

encoder, ENC_DIM = build_encoder()

class MaskNet(nn.Module):
    def __init__(self, enc, enc_dim, n_freq=N_FFT//2+1, hid=256, proj=ENC_PROJ):
        super().__init__()
        self.enc = enc; self.use_enc = enc is not None and enc_dim > 0
        if self.use_enc:
            self.enc_proj = nn.Linear(enc_dim, proj)
            self.film1 = nn.Linear(proj, 2*hid); self.film2 = nn.Linear(proj, 4*hid)
        self.inp = nn.Linear(n_freq, hid)
        self.rnn = nn.GRU(hid, hid, 2, batch_first=True, bidirectional=True, dropout=0.1)
        self.mid = nn.Linear(2*hid, 2*hid); self.out = nn.Linear(2*hid, n_freq)
    def forward(self, wav, mag_log, T):
        h = self.inp(mag_log)
        if self.use_enc:
            e = self.enc_proj(self.enc(wav))
            e = F.interpolate(e.transpose(1, 2), size=T, mode="linear",
                              align_corners=False).transpose(1, 2)
            g, b_ = self.film1(e).chunk(2, -1); h = h*(1+g)+b_
            h, _ = self.rnn(h)
            g, b_ = self.film2(e).chunk(2, -1); h = h*(1+g)+b_
        else:
            h, _ = self.rnn(h)
        return torch.sigmoid(self.out(F.relu(self.mid(h))))

def enhance_batch(model, mix):
    X = stft(mix)
    m = model(mix, torch.log1p(X.abs()).transpose(1, 2), X.shape[-1]).transpose(1, 2)
    return istft(X*m, mix.shape[-1]), m

def si_sdr_loss(est, ref, eps=1e-8):
    est = est-est.mean(-1, keepdim=True); ref = ref-ref.mean(-1, keepdim=True)
    a = (est*ref).sum(-1, keepdim=True)/((ref*ref).sum(-1, keepdim=True)+eps)
    t = a*ref; n = est-t
    return -(10*torch.log10(((t**2).sum(-1)+eps)/((n**2).sum(-1)+eps))).mean()

model = MaskNet(encoder, ENC_DIM).to(device)
print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M trainable")
with torch.no_grad(): est, m = enhance_batch(model, b["mix"].to(device))
assert est.shape == b["mix"].shape and 0 <= float(m.min()) and float(m.max()) <= 1
assert torch.isfinite(est).all()
print("shapes and mask range OK")

---
## 6. Train on simulated mixtures

The loss is **SI-SDR in the time domain** — the metric itself. The mask is applied to the
complex STFT and inverted, so gradients flow from the waveform back through the mask.

The assertion at the end requires the model to beat the identity function by at least 0.5 dB.
A model that loses to doing nothing should never reach a submission.

In [ ]:
@torch.no_grad()
def eval_sisdr(model, dl):
    model.eval(); ins, outs = [], []
    for batch in dl:
        mix, clean = batch["mix"].to(device), batch["clean"].to(device)
        est, _ = enhance_batch(model, mix)
        for e, c, m_ in zip(est.float().cpu().numpy(), clean.cpu().numpy(), mix.cpu().numpy()):
            ins.append(si_sdr(c, m_)); outs.append(si_sdr(c, e))
    return float(np.mean(ins)), float(np.mean(outs))

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*max(1, len(train_dl)))
best = -1e9
for ep in range(EPOCHS):
    model.train()
    if encoder is not None: encoder.eval()
    tot = n = 0; t0 = time.time()
    for batch in train_dl:
        mix, clean = batch["mix"].to(device), batch["clean"].to(device)
        est, _ = enhance_batch(model, mix)
        loss = si_sdr_loss(est, clean)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(params, 5.0); opt.step(); sched.step()
        tot += float(loss.detach()); n += 1
    si, so = eval_sisdr(model, val_dl)
    print(f"ep{ep:02d} [{(time.time()-t0)/60:.1f}m] loss {tot/max(1,n):.3f} | "
          f"SI-SDR {si:.2f} -> {so:.2f} ({so-si:+.2f} dB)")
    if so-si > best:
        best = so-si; torch.save({"model": model.state_dict(), "si_sdri": best}, CKPT)
print(f"\nbest SI-SDRi {best:+.2f} dB")
assert best > 0.5, "model does not beat the identity function - raise N_MIX or EPOCHS"

---
## 7. Output stage

Three guards, then a check on every one.

1. **NaN/Inf are zeroed** so one bad frame cannot poison a file.
2. **Blend with the mixture**: `y = a*est + (1-a)*mix` caps how much the model can damage
   speech, for a fraction of a dB.
3. **RMS restoration** — the important one. A sigmoid mask only attenuates, and nothing in an
   SI-SDR loss penalises an output 30 dB too quiet. SI-SDR is scale-invariant; ASR is not, and
   the transcripts you submit come from this audio.

The cell prints `rms_ratio` (should be ~1.0) and confirms SI-SDR is **identical** with and
without the restoration — which is the proof that the fix is free.

Long clips are processed in overlapping windows with cross-fades, so an 11-hour test set does
not OOM partway through.

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=False)["model"])
model.eval()
MAX_ONESHOT = 30.0*CFG["sr"]

@torch.no_grad()
def _enh_core(wav, blend, match_rms):
    x = torch.from_numpy(np.asarray(wav, dtype=np.float32)).unsqueeze(0).to(device)
    est, _ = enhance_batch(model, x)
    y = est[0].float().cpu().numpy()
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    if len(y) < len(wav): y = np.pad(y, (0, len(wav)-len(y)))
    y = y[:len(wav)]
    y = blend*y + (1.0-blend)*wav
    if match_rms:
        r_in = float(np.sqrt((wav**2).mean())); r_out = float(np.sqrt((y**2).mean()))
        if r_out > 1e-8 and r_in > 1e-8: y = y*(r_in/r_out)
    pk = float(np.abs(y).max())
    if pk > 0.99: y = y/pk*0.99
    return y.astype(np.float32)

def enhance(wav, blend=BLEND, match_rms=MATCH_RMS):
    wav = np.asarray(wav, dtype=np.float32)
    if len(wav) <= MAX_ONESHOT: return _enh_core(wav, blend, match_rms)
    win, ov = int(MAX_ONESHOT), int(1.0*CFG["sr"])
    out = np.zeros(len(wav), np.float32); acc = np.zeros(len(wav), np.float32)
    for s0 in range(0, len(wav), win-ov):
        seg = wav[s0:s0+win]
        if len(seg) < CFG["sr"]*0.1: break
        y = _enh_core(seg, blend, match_rms)
        fade = np.ones(len(y), np.float32); k = min(ov, len(y)//2)
        if k > 0 and s0 > 0: fade[:k] = np.linspace(0, 1, k)
        if k > 0 and s0+win < len(wav): fade[-k:] = np.linspace(1, 0, k)
        out[s0:s0+len(y)] += y*fade; acc[s0:s0+len(y)] += fade
    return (out/np.maximum(acc, 1e-6)).astype(np.float32)

def check_output(wav_in, wav_out):
    r_in = float(np.sqrt((wav_in**2).mean())); r_out = float(np.sqrt((wav_out**2).mean()))
    p = dict(finite=bool(np.isfinite(wav_out).all()),
             len_match=len(wav_out) == len(wav_in),
             dtype_f32=wav_out.dtype == np.float32, mono=wav_out.ndim == 1,
             rms_in=r_in, rms_out=r_out, rms_ratio=r_out/(r_in+1e-12),
             peak=float(np.abs(wav_out).max()))
    p["level_ok"] = 0.5 <= p["rms_ratio"] <= 2.0
    p["not_silent"] = r_out > 1e-4
    p["no_clip"] = p["peak"] <= 0.999
    p["ok"] = all(p[k] for k in ["finite", "len_match", "dtype_f32", "mono",
                                 "level_ok", "not_silent", "no_clip"])
    return p

fails = 0; ratios = []
for r in sim_val_recs[:40]:
    mix, _ = sf.read(r["mix"], dtype="float32")
    p = check_output(mix, enhance(mix)); ratios.append(p["rms_ratio"])
    fails += (not p["ok"])
print(f"output checks {40-fails}/40 pass | rms ratio min {min(ratios):.3f} "
      f"median {np.median(ratios):.3f} max {max(ratios):.3f}")
assert fails == 0
a = [si_sdr(sf.read(r['clean'],dtype='float32')[0],
            enhance(sf.read(r['mix'],dtype='float32')[0], match_rms=False))
     for r in sim_val_recs[:40]]
c = [si_sdr(sf.read(r['clean'],dtype='float32')[0],
            enhance(sf.read(r['mix'],dtype='float32')[0], match_rms=True))
     for r in sim_val_recs[:40]]
print(f"SI-SDR without rms-match {np.mean(a):+.2f} | with {np.mean(c):+.2f} (identical)")

---
## 8. The mandated ASR

`ARTPARK-IISc/SraVaani-1.0`, unchanged — the audit re-runs it on a hidden subset of your WAVs
and compares against your transcripts.

It is loaded from the **local snapshot path**, because their loader does
`os.path.join(path, "tokenizer.model")` and so fails when handed a bare repo id.

In [ ]:
import tempfile
from huggingface_hub import snapshot_download
from transformers import AutoModel as _AM

ASR_PATH = snapshot_download(ASR_REPO, token=HF_TOKEN if "HF_TOKEN" in globals() else None)
asr_model = _AM.from_pretrained(ASR_PATH, trust_remote_code=True).to(device).eval()
print("mandated ASR loaded:", ASR_PATH)

def _tmp(wavs):
    ps = []
    for w in wavs:
        f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        sf.write(f.name, np.asarray(w, dtype=np.float32), CFG["sr"], subtype=WAV_SUBTYPE)
        ps.append(f.name)
    return ps

@torch.no_grad()
def transcribe_paths(paths):
    try:
        hyps = asr_model.transcribe(paths, return_hypotheses=True)
        return [(h.text if hasattr(h, "text") else str(h)).strip() for h in hyps]
    except Exception:
        out = []
        for p in paths:
            try:
                h = asr_model.transcribe([p], return_hypotheses=True)[0]
                out.append((h.text if hasattr(h, "text") else str(h)).strip())
            except Exception:
                out.append("")
        return out

def transcribe_many(wavs):
    ps = _tmp(wavs)
    try: return transcribe_paths(ps)
    finally:
        for p in ps:
            try: os.remove(p)
            except Exception: pass

def transcribe(w): return transcribe_many([w])[0]

t = transcribe(sf.read(sim_val_recs[0]["clean"], dtype="float32")[0])
print("smoke transcript:", repr(t[:120]))
assert isinstance(t, str), "ASR did not return text"

---
## 9. Score on the validation set

Sampling is **stratified**. `val_items` is natural-first, so a plain `[:N]` slice would contain
no synthetic clips at all and SI-SDR would silently never be computed. Each pool is sampled
explicitly, preferring clips that carry what each metric needs.

Reference and noisy transcriptions are computed once, since only the enhanced side changes as
the blend varies.

`pooled WER(noisy)` is the organisers' fixed baseline recomputed locally — if it lands near
0.31, the local setup matches theirs and the sweep is trustworthy.

In [ ]:
# ============ score the validation set with the OFFICIAL scorer ============
# Stratified: val_items is natural-first, so a plain [:N] slice would contain no synthetic
# clips at all -> SI-SDR silently never computed. Sample each pool explicitly.
N_SYN, N_NAT = 60, 60
syn_pool = [v for v in val_items if v["syn"] and v["ref"] is not None]
nat_pool = [v for v in val_items if not (v["syn"] and v["ref"] is not None)]
syn_pool.sort(key=lambda v: v["gt"] is None)
nat_pool.sort(key=lambda v: v["gt"] is None)
subset = syn_pool[:N_SYN] + nat_pool[:N_NAT]
print(f"pools: synthetic-with-ref {len(syn_pool)} | other {len(nat_pool)}")
print(f"subset: {len(subset)} clips "
      f"({sum(1 for v in subset if v['ref'] is not None)} scorable for SI-SDR, "
      f"{sum(1 for v in subset if v['gt'])} with GT transcript)")
assert subset, "empty subset - check the validation loader output above"
if not syn_pool:
    print("!! no synthetic clip matched a clean reference -> SI-SDR cannot be computed")

gt_map, noisy_map, ids, cache, refcache = {}, {}, [], {}, {}
for v in tqdm(subset, desc="decode + reference/noisy ASR"):
    cache[v["id"]] = read_audio(v["path"])
    if v["ref"] is not None:
        refcache[v["id"]] = read_audio(v["ref"])
    g = v["gt"] or (transcribe(refcache[v["id"]]) if v["id"] in refcache else None)
    if not g:
        continue
    ids.append(v["id"]); gt_map[v["id"]] = g
    noisy_map[v["id"]] = transcribe(cache[v["id"]])

print(f"\nusable for dWER : {len(ids)}/{len(subset)}")
print(f"usable for SI-SDR: {len(refcache)}/{len(subset)}")
if ids:
    wn = jiwer_wer([normalize(gt_map[i]) for i in ids],
                   [normalize(noisy_map[i]) or "@" for i in ids])
    print(f"pooled WER(noisy) = {wn:.3f}   <- the organizers' fixed baseline is ~this")

def score_val(blend):
    enh, sis = {}, []
    for v in subset:
        y = enhance(cache[v["id"]], blend)
        if v["id"] in gt_map:
            enh[v["id"]] = transcribe(y)
        if v["id"] in refcache:                      # SI-SDR: synthetic subset only
            sis.append(si_sdr(refcache[v["id"]], y))
    dw = delta_wer(gt_map, noisy_map, enh, ids) if ids else float("nan")
    return (float(np.mean(sis)) if sis else float("nan")), dw, len(sis), len(ids)

print("\nblend sweep (Combined = SI-SDR + 100 x dWER):")
rows = []
for bl in [0.3, 0.5, 0.7, 0.85, 1.0]:
    s, dw, n_s, n_d = score_val(bl)
    comb = (0.0 if math.isnan(s) else s) + (0.0 if math.isnan(dw) else 100*dw)
    rows.append((bl, s, dw, comb))
    print(f"  blend {bl:.2f}: SI-SDR {s:+7.2f} (n={n_s:>3})  "
          f"dWER {100*dw:+6.2f}% (n={n_d:>3})  Combined {comb:+7.2f}")
valid = [r for r in rows if not (math.isnan(r[1]) and math.isnan(r[2]))]
assert valid, "neither metric could be computed"
BEST_BLEND = max(valid, key=lambda r: r[3])[0]
print(f"\nbest blend {BEST_BLEND}")
if all(math.isnan(r[1]) for r in rows):
    print("!! SI-SDR nan everywhere - blend chosen on dWER alone, but the leaderboard is "
          "dominated by SI-SDR. Fix reference matching first.")

---
## 10. Fine-tune on the real validation pairs

This is the step that closes the domain gap.

Our simulated mixtures gave +5 dB SI-SDRi in training but only ~+1.3 dB on the organisers'
synthetic clips. The cause is the clean bank: Vaani speech with no *annotated* event still
contains real background, so the model learned to preserve exactly the noise it should remove.

`syntheticNoiseAudio/` + `syntheticCleanRefAudio/` are genuinely paired data from the
organisers' own generator — about an hour of precisely the right distribution. We fine-tune on
it at a lower learning rate, holding out 15% so model selection stays honest.

Two cautions. Once you train on validation you can no longer use all of it for selection, so the
blend sweep afterwards runs on the **held-out slice only**. And an hour is a small set — if the
held-out number diverges from the training loss, cut epochs or mix simulated data back in at a
low weight.

Set `DO_FINETUNE = False` to skip and keep the simulation-only model.

In [ ]:
DO_FINETUNE = True
FT_EPOCHS, FT_LR, FT_HOLD = 30, 3e-4, 0.15

pairs = [v for v in val_items if v["ref"] is not None]
print(f"{len(pairs)} real (noisy, clean) pairs available")

if DO_FINETUNE and len(pairs) >= 20:
    random.Random(0).shuffle(pairs)
    n_hold = max(20, int(FT_HOLD*len(pairs)))
    hold_items, fit_items = pairs[:n_hold], pairs[n_hold:]
    print(f"  {len(fit_items)} fine-tune / {len(hold_items)} held out")

    class RealPairs(Dataset):
        def __init__(self, items, train=True): self.items, self.train = items, train
        def __len__(self): return len(self.items)
        def __getitem__(self, i):
            v = self.items[i]
            mix, clean = read_audio(v["path"]), read_audio(v["ref"])
            n = min(len(mix), len(clean)); mix, clean = mix[:n], clean[:n]
            if n > N_SAMP:
                s = random.randint(0, n-N_SAMP) if self.train else 0
                mix, clean = mix[s:s+N_SAMP], clean[s:s+N_SAMP]
            elif n < N_SAMP:
                mix = np.pad(mix, (0, N_SAMP-n)); clean = np.pad(clean, (0, N_SAMP-n))
            return dict(mix=torch.from_numpy(mix.astype(np.float32)),
                        clean=torch.from_numpy(clean.astype(np.float32)))

    ft_dl = DataLoader(RealPairs(fit_items), batch_size=BS, shuffle=True,
                       num_workers=2, drop_last=True)
    hold_dl = DataLoader(RealPairs(hold_items, train=False), batch_size=BS,
                         shuffle=False, num_workers=2)

    prm = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(prm, lr=FT_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS*max(1, len(ft_dl)))
    si0, so0 = eval_sisdr(model, hold_dl)
    print(f"before fine-tune: held-out SI-SDR {si0:+.2f} -> {so0:+.2f} ({so0-si0:+.2f} dB)")

    best_abs = so0
    for ep in range(FT_EPOCHS):
        model.train()
        if encoder is not None: encoder.eval()
        tot = n = 0
        for batch in ft_dl:
            mix, clean = batch["mix"].to(device), batch["clean"].to(device)
            est, _ = enhance_batch(model, mix)
            loss = si_sdr_loss(est, clean)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(prm, 5.0); opt.step(); sched.step()
            tot += float(loss.detach()); n += 1
        si, so = eval_sisdr(model, hold_dl)
        print(f"ft ep{ep:02d} loss {tot/max(1,n):.3f} | held-out SI-SDR "
              f"{si:+.2f} -> {so:+.2f} ({so-si:+.2f} dB)")
        if so > best_abs:
            best_abs = so
            torch.save({"model": model.state_dict(), "si_sdr": so}, WORK/"t2_model_real.pt")
    if (WORK/"t2_model_real.pt").exists():
        model.load_state_dict(torch.load(WORK/"t2_model_real.pt",
                                         weights_only=False)["model"])
    model.eval()
    print(f"\nbest held-out ABSOLUTE SI-SDR {best_abs:+.2f} dB  <- the leaderboard reports this")

    # re-sweep the blend on the held-out slice only
    subset = hold_items
    gt_map, noisy_map, ids, cache, refcache = {}, {}, [], {}, {}
    for v in tqdm(subset, desc="held-out ASR"):
        cache[v["id"]] = read_audio(v["path"]); refcache[v["id"]] = read_audio(v["ref"])
        g = v["gt"] or transcribe(refcache[v["id"]])
        if not g: continue
        ids.append(v["id"]); gt_map[v["id"]] = g
        noisy_map[v["id"]] = transcribe(cache[v["id"]])
    print("\nblend sweep on the HELD-OUT slice:")
    rows = []
    for bl in [0.3, 0.5, 0.7, 0.85, 1.0]:
        s, dw, n_s, n_d = score_val(bl)
        comb = (0.0 if math.isnan(s) else s) + (0.0 if math.isnan(dw) else 100*dw)
        rows.append((bl, s, dw, comb))
        print(f"  blend {bl:.2f}: SI-SDR {s:+7.2f} (n={n_s:>3})  "
              f"dWER {100*dw:+6.2f}% (n={n_d:>3})  Combined {comb:+7.2f}")
    BEST_BLEND = max(rows, key=lambda r: r[3])[0]
    print(f"\nbest blend {BEST_BLEND}")
else:
    print("skipping fine-tune (DO_FINETUNE=False or too few pairs)")

---
## 11. Build the submission

Exactly per spec: WAVs as 16 kHz mono **PCM-16** named `<clip_id>.wav`, plus a single
`transcripts.jsonl`, all at the ZIP **root**.

The transcripts are produced by running the mandated ASR over the enhanced files **already on
disk**, so what you submit is provably what the audit will re-run.

Hard assertions before an archive exists: one WAV per test clip (a missing file costs −50 dB),
`clip_id` values matching the WAV stems exactly, `{clip_id, text}` and nothing else, fewer than
25% empty transcripts, and root-level placement.

In [ ]:
# TEST_DIR: prefer an explicit mount, else the biggest audio dir that is NOT the validation set
TEST_DIR = globals().get("TEST_DIR", None)
if TEST_DIR is None:
    cand = Path("/kaggle/input/indoml-track2-test")
    TEST_DIR = cand if cand.exists() else biggest_audio_dir(
        exclude=(VAL_DIR,) if "VAL_DIR" in globals() and VAL_DIR else ())
print("TEST_DIR =", TEST_DIR)
assert TEST_DIR is not None, "attach the Track 2 test audio"

test_files = sorted([p for p in TEST_DIR.rglob("*") if p.suffix.lower() in AUD])
stems = [p.stem for p in test_files]
dupes = [x for x, c in Counter(stems).items() if c > 1]
assert not dupes, f"duplicate stems: {dupes[:5]}"
approx_h = (sum(sf.info(str(p)).duration for p in test_files[:200]) /
            min(200, len(test_files))*len(test_files)/3600)
print(f"{len(test_files)} clips, ~{approx_h:.1f} h (expect ~11 h)")

SUB_DIR = WORK/"t2_submission"; SUB_DIR.mkdir(parents=True, exist_ok=True)
stats, reused = [], 0
for p in tqdm(test_files, desc="enhance"):
    dst = SUB_DIR/f"{p.stem}.wav"; w = read_audio(p)
    if dst.exists():
        y, _ = sf.read(str(dst), dtype="float32"); reused += 1
    else:
        y = enhance(w, BEST_BLEND)
        sf.write(dst, y, CFG["sr"], subtype=WAV_SUBTYPE)
    stats.append(check_output(w, y))
print(f"reused {reused}")

n_bad = sum(1 for s in stats if not s["ok"]); rr = [s["rms_ratio"] for s in stats]
print(f"\naudio: {len(stats)} files, {n_bad} failing | rms ratio "
      f"min {min(rr):.3f} median {np.median(rr):.3f} max {max(rr):.3f}")
print(f"  silent {sum(1 for s in stats if not s['not_silent'])} | "
      f"non-finite {sum(1 for s in stats if not s['finite'])} | "
      f"clipped {sum(1 for s in stats if not s['no_clip'])} | "
      f"len mismatch {sum(1 for s in stats if not s['len_match'])}")
assert n_bad == 0, "outputs failed the audio checks"
assert len(list(SUB_DIR.glob('*.wav'))) == len(test_files), "MISSING WAV = -50 dB"

In [ ]:
TJ = WORK/TRANSCRIPT_NAME
done = {}
if TJ.exists():
    for line in open(TJ, encoding="utf-8"):
        try:
            o = json.loads(line); done[o[ID_KEY]] = o[TEXT_KEY]
        except Exception: pass
    print(f"resuming: {len(done)} transcripts present")

BATCH = 8
todo = [p for p in test_files if p.stem not in done]
for i in tqdm(range(0, len(todo), BATCH), desc="transcribe"):
    chunk = todo[i:i+BATCH]
    texts = transcribe_paths([str(SUB_DIR/f"{p.stem}.wav") for p in chunk])
    for p, t in zip(chunk, texts): done[p.stem] = (t or "").strip()
    if (i//BATCH) % 25 == 0:
        with open(TJ, "w", encoding="utf-8") as f:
            for s_ in stems:
                if s_ in done:
                    f.write(json.dumps({ID_KEY: s_, TEXT_KEY: done[s_]},
                                       ensure_ascii=False)+"\n")

with open(TJ, "w", encoding="utf-8") as f:
    for s_ in stems:
        f.write(json.dumps({ID_KEY: s_, TEXT_KEY: done.get(s_, "")}, ensure_ascii=False)+"\n")

lines = [json.loads(l) for l in open(TJ, encoding="utf-8")]
empty = sum(1 for r in lines if not r[TEXT_KEY])
print(f"\ntranscripts: {len(lines)} lines | empty {empty} ({100*empty/len(lines):.1f}%)")
print("sample:", json.dumps(lines[0], ensure_ascii=False)[:170])
assert len(lines) == len(test_files), "one line per clip required"
assert {r[ID_KEY] for r in lines} == set(stems), "clip_ids must match the wav stems"
assert set(lines[0].keys()) == {ID_KEY, TEXT_KEY}, "unexpected keys in transcripts.jsonl"
assert empty < 0.25*len(lines), (f"{empty} empty transcripts -> WER 1.0 each. This is exactly "
                                 "what produced -69.35.")

In [ ]:
zp = WORK/"submission_track2.zip"
if zp.exists(): zp.unlink()
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(TJ, TRANSCRIPT_NAME)                       # ROOT
    for f_ in sorted(SUB_DIR.glob("*.wav")):
        z.write(f_, f_.name)                           # ROOT
names = zipfile.ZipFile(zp).namelist()
print(f"zip: {len(names)} entries = {len(names)-1} wav + 1 jsonl | "
      f"root-level {all('/' not in n for n in names)} | {zp.stat().st_size/1e6:.0f} MB")
assert TRANSCRIPT_NAME in names, "transcripts.jsonl missing -> dWER fully penalised"
assert all("/" not in n for n in names), "files must be at the ZIP root"
assert set(names) == {f"{s}.wav" for s in stems} | {TRANSCRIPT_NAME}
i_ = sf.info(str(next(SUB_DIR.glob("*.wav"))))
print(f"first wav: {i_.samplerate} Hz, {i_.channels} ch, subtype {i_.subtype}")
assert i_.samplerate == 16000 and i_.channels == 1 and i_.subtype == "PCM_16"
print(f"\nREADY: {zp}")

---
## 12. Reading the result

**Combined is dominated by SI-SDR.** The leader is at 16.14 with dWER +0.44. Keep dWER at or
above zero and push SI-SDR.

| symptom | cause |
|---|---|
| dWER near −69 | `transcripts.jsonl` missing or empty |
| dWER a few points negative | the enhancement genuinely hurts the ASR — lower `BLEND` |
| SI-SDR negative but improving | the input itself is around −1.4 dB; the leaderboard reports **absolute** SI-SDR, so track the absolute number |
| Audit = 0 | transcripts do not match the mandated ASR on your WAVs |

**Next gains, in order.** Raise `BUDGET` and `FT_EPOCHS`. Predict a **complex** mask instead of a
magnitude one — reconstructing with the mixture phase caps achievable SI-SDR. Generate more
training data with the organisers' own recipe once you can measure it. Try `T2_ENCODER = "none"`
as a control: if it matches, the encoder is not earning its runtime.

**State plainly in the report** that the simulated clean reference is Vaani speech with no
*annotated* event, not truly clean audio — and that the final model is fine-tuned on the
released validation pairs, with a held-out slice used for selection.